# Дообучение REVE на NMT → проверка на турецком наборе

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-21 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Цель** | Дообучить REVE (`brain-bzh/reve-base`) по той же схеме, что CBraMod и DeepEpilepsy, и сравнить / собрать в ансамбль |
| **Данные** | Обучение: NMT train (`mhmdrdwn/nmt-eeg`), до 8 окон на запись, 15 % записей — выбор эпохи. Проверка один раз в конце: NMT eval и турецкий набор (`buraktaci/turkish-epilepsy`) |
| **Вход** | 200 Гц, 19 каналов, окна 10 с (см. выводы); каждое окно приводится к общему СКО по всем каналам (абсолютная амплитуда пропадает) |
| **Аугментации** | шум 0,1·СКО, зануление 1–2 каналов (p = 0,5), зеркальная замена полушарий (p = 0,5), сдвиг до 2 с |
| **Обучение** | основа lr 1·10⁻⁵, голова 5·10⁻⁴, косинус, пакет 32, fp16 с проверкой NaN, до 3 запусков, лимит 85 мин |
| **Среда** | Kaggle, GPU T4 |


In [ ]:
import os, sys, glob, time
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, mne, scipy.io, scipy.signal as ss
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata, spearmanr
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel
mne.set_log_level("ERROR")
DEV = "cuda" if torch.cuda.is_available() else "cpu"
T_NB = time.time()

SFREQ, SEG = 200, 10
L = SFREQ * SEG
CAP_EVAL = 60          # окон на запись для NMT eval и Турции (равномерно по записи), одинаково для всех запусков

EL = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]
NEW = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8"}
pos_bank = AutoModel.from_pretrained("brain-bzh/reve-positions", trust_remote_code=True)
try:
    POS = pos_bank([NEW.get(e, e) for e in EL])
except Exception as e:
    print("новые названия не подошли:", e); POS = pos_bank(EL)
POS = POS.to(DEV)
print("позиции:", tuple(POS.shape))

MIRROR_PAIRS = {"Fp1": "Fp2", "F3": "F4", "C3": "C4", "P3": "P4", "O1": "O2", "F7": "F8", "T3": "T4", "T5": "T6"}
MIRROR_PAIRS.update({v: k for k, v in list(MIRROR_PAIRS.items())})
MIRROR = [EL.index(MIRROR_PAIRS.get(e, e)) for e in EL]

def rv_input(W):
    """Окно → общее СКО 1 по всем каналам и отсчётам (абсолютная амплитуда пропадает, соотношение каналов остаётся), обрезка ±15."""
    x = W.astype(np.float32)
    x = x - x.mean(-1, keepdims=True)
    x = x / (x.std(axis=(1, 2), keepdims=True) + 1e-6)
    return torch.from_numpy(np.clip(x, -15, 15))

class RVNet(nn.Module):
    """REVE (22 слоя) + голова: среднее по времени с сохранением каналов -> 19*512 -> два линейных слоя."""
    def __init__(self, dropout=0.3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained("brain-bzh/reve-base", trust_remote_code=True)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(19 * 512, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 1))
    def forward(self, x):
        o = self.backbone(x, POS.expand(x.size(0), -1, -1))
        if isinstance(o, (tuple, list)): o = o[0]
        if hasattr(o, "last_hidden_state"): o = o.last_hidden_state
        assert o.dim() == 4 and o.shape[1] == 19, tuple(o.shape)
        return self.head(o.float().mean(2).flatten(1)).squeeze(-1)

def new_rv(seed):
    torch.manual_seed(seed)
    return RVNet().to(DEV)

AMP = dict(device_type="cuda", dtype=torch.float16, enabled=DEV == "cuda")

def predict(model, Ws, bs=64):
    """Медиана вероятности по окнам для каждой записи."""
    model.eval(); out = []
    with torch.no_grad(), torch.autocast(**AMP):
        for W in Ws:
            p = []
            for b in range(0, len(W), bs):
                p.append(torch.sigmoid(model(rv_input(W[b:b + bs]).to(DEV)).float()).cpu().numpy())
            out.append(np.median(np.concatenate(p)))
    return np.array(out)

def rank(s): return rankdata(s) / len(s)
def boot_ci(yt, s, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(yt), len(yt))
        if len(set(yt[i])) == 2: v.append(roc_auc_score(yt[i], s[i]))
    return np.percentile(v, [2.5, 97.5])

SOS_BP = ss.butter(4, [0.3, 75], btype="bandpass", fs=SFREQ, output="sos")
B_N, A_N = ss.iirnotch(50, 30, fs=SFREQ)
def cb_filter(X):
    """0,3-75 Гц + режекция 50 Гц, как в блокноте CBraMod."""
    if X.shape[-1] < 60 * SFREQ:
        return X
    return ss.filtfilt(B_N, A_N, ss.sosfiltfilt(SOS_BP, X.astype(np.float64), axis=-1), axis=-1)

def windows(X):
    n = X.shape[1] // L
    return X[:, :n * L].reshape(19, n, L).transpose(1, 0, 2)

def spread(n, k):
    return np.unique(np.linspace(0, n - 1, k).astype(int)) if n > k else np.arange(n)
print("устройство:", torch.cuda.get_device_name(0) if DEV == "cuda" else "процессор")

## 2. NMT

In [ ]:
ROOT_NMT = glob.glob("/kaggle/input/**/nmt_scalp_eeg_dataset", recursive=True)[0]
lab = pd.read_csv(f"{ROOT_NMT}/Labels.csv")
paths = {os.path.basename(p): p for p in glob.glob(f"{ROOT_NMT}/**/*.edf", recursive=True)}
lab["path"] = lab["recordname"].map(paths)
lab = lab.dropna(subset=["path"]).sort_values("recordname").reset_index(drop=True)
lab["y"] = (lab["label"].str.lower() == "abnormal").astype(int)
print(pd.crosstab(lab["loc"], lab["label"]))

def warm(p):
    with open(p, "rb") as f:
        while f.read(1 << 24):
            pass
t0 = time.time()
with ThreadPoolExecutor(32) as ex:
    for i, _ in enumerate(ex.map(warm, lab.path)):
        if (i + 1) % 400 == 0 or i + 1 == len(lab):
            print(f"NMT подкачано {i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)

ALIAS = {"T7": "T3", "T8": "T4", "P7": "T5", "P8": "T6"}
TARGET = [e.upper() for e in EL]
def norm(ch):
    c = ch.upper().replace("EEG", "").strip()
    for s in ["-REF", "-AVG", "-LE", "-AR"]:
        c = c.replace(s, "")
    return ALIAS.get(c.strip(" -_"), c.strip(" -_"))

NMT_STORE_K = 100.0   # масштаб в файлах неверный; окна всё равно нормируются, множитель только для float16
def nmt_job(i):
    r = lab.iloc[i]
    try:
        raw = mne.io.read_raw_edf(r.path, preload=True)
        if raw.info["sfreq"] != SFREQ:
            raw.resample(SFREQ)
        names = {norm(c): c for c in raw.ch_names}
        X = raw.get_data(picks=[names[t] for t in TARGET]) * 1e6 * NMT_STORE_K
    except Exception:
        return r.recordname, None
    W = windows(cb_filter(X))
    if len(W) == 0:
        return r.recordname, None
    if r["loc"] == "train" and len(W) > 8:
        W = W[np.sort(np.random.default_rng(i).choice(len(W), 8, replace=False))]
    elif r["loc"] == "eval":
        W = W[spread(len(W), CAP_EVAL)]
    return r.recordname, W.astype(np.float16)

NMT = {}
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for i, (name, res) in enumerate(ex.map(nmt_job, range(len(lab)))):
        if res is not None:
            NMT[name] = res
        if (i + 1) % 400 == 0 or i + 1 == len(lab):
            print(f"NMT прочитано {i+1}/{len(lab)} {time.time()-t0:.0f} с", flush=True)
lab = lab[lab.recordname.isin(NMT)].reset_index(drop=True)
tr, ev = lab[lab["loc"] == "train"].reset_index(drop=True), lab[lab["loc"] == "eval"].reset_index(drop=True)
print(pd.crosstab(lab["loc"], lab["label"]))
print("окон: train", sum(len(NMT[n]) for n in tr.recordname), "| eval", sum(len(NMT[n]) for n in ev.recordname))

## 3. Турецкий набор: 19 каналов, средний референт, 200 Гц, окна 10 с подряд

In [ ]:
ROOT_TR = glob.glob("/kaggle/input/**/Turkish Epilepsy Dataset", recursive=True)[0]
tfiles = [(f, 1) for f in sorted(glob.glob(f"{ROOT_TR}/epilepsy/*.mat"))] + \
         [(f, 0) for f in sorted(glob.glob(f"{ROOT_TR}/normal/*.mat"))]
t0 = time.time()
with ThreadPoolExecutor(16) as ex:
    for i, _ in enumerate(ex.map(warm, [f for f, _ in tfiles])):
        if (i + 1) % 40 == 0 or i + 1 == len(tfiles):
            print(f"Турция подкачано {i+1}/{len(tfiles)} {time.time()-t0:.0f} с", flush=True)
# индексы турецких каналов в порядке EL: Fp1 Fp2 F3 F4 C3 C4 P3 P4 O1 O2 F7 F8 T3 T4 T5 T6 Fz Cz Pz
IDX = [0, 1, 2, 3, 5, 6, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 4, 7, 10]

def tr_job(f):
    d = scipy.io.loadmat(f)["data"]
    X = np.stack([np.stack([d[s, c].ravel() for c in IDX]) for s in range(d.shape[0])])   # (куски, 19, 7500)
    X = X - X.mean(1, keepdims=True)                                                       # средний референт
    X = ss.resample_poly(X, 2, 5, axis=-1)                                                  # 200 Гц
    W = windows(cb_filter(np.concatenate(list(X), axis=-1)))
    return W[spread(len(W), CAP_EVAL)].astype(np.float16), float(np.std(X))

TR, TR_AMP = [], []
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for a, b in ex.map(tr_job, [f for f, _ in tfiles]):
        TR.append(a); TR_AMP.append(b)
y_tr = np.array([yy for _, yy in tfiles]); tr_ids = [os.path.basename(f) for f, _ in tfiles]
TR_AMP = np.array(TR_AMP)
print(f"Турция прочитано за {time.time()-t0:.0f} с | эпилепсия {int(y_tr.sum())}, здоровые {int((1-y_tr).sum())} | окон на запись: медиана {int(np.median([len(w) for w in TR]))}")
print("эталон перекоса: СКО сигнала записи как оценка, ROC AUC =", round(roc_auc_score(y_tr, TR_AMP), 3))

## 4. Дообучение REVE

In [ ]:
from sklearn.model_selection import train_test_split

fit_names, val_names = train_test_split(tr.recordname.to_numpy(), test_size=0.15, stratify=tr.y, random_state=0)
ymap = dict(zip(lab.recordname, lab.y))
yw = np.concatenate([np.full(len(NMT[n]), ymap[n]) for n in fit_names])
y_val = np.array([ymap[n] for n in val_names])
w_cls = np.where(yw == 1, 0.5 / (yw == 1).sum(), 0.5 / (yw == 0).sum()); w_cls /= w_cls.sum()
Xw = np.concatenate([NMT[n] for n in fit_names])
val_W = [NMT[n] for n in val_names]
print(f"окон для обучения: {len(yw)} (патология {yw.mean():.2f}) | записей на выбор эпохи: {len(val_names)}")

NOISE, BS, EPOCHS = 0.1, 32, 6
def augment(x, g):
    B = x.shape[0]
    x = x + NOISE * torch.randn_like(x)
    m = torch.rand(B, device=x.device) < 0.5
    x[m] = x[m][:, MIRROR]
    for b in range(B):
        if g.random() < 0.5:
            x[b, g.choice(19, g.integers(1, 3), replace=False)] = 0
        x[b] = torch.roll(x[b], int(g.integers(-2 * SFREQ, 2 * SFREQ + 1)), dims=-1)
    return x

def make_opt(model, total_steps):
    back = [p for n, p in model.named_parameters() if n.startswith("backbone")]
    head = [p for n, p in model.named_parameters() if not n.startswith("backbone")]
    opt = torch.optim.AdamW([{"params": back, "lr": 1e-5}, {"params": head, "lr": 5e-4}], weight_decay=0.05)
    warm_n = max(1, int(0.05 * total_steps))
    f = lambda s: (s + 1) / warm_n if s < warm_n else 0.5 * (1 + np.cos(np.pi * (s - warm_n) / max(1, total_steps - warm_n))) * 0.99 + 0.01
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, f)

def step(model, opt, sched, scaler, x, t):
    with torch.autocast(**AMP):
        out = model(x)
    loss = F.binary_cross_entropy_with_logits(out.float(), t.float())
    bad = not torch.isfinite(loss)
    opt.zero_grad(); scaler.scale(loss).backward()
    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt); scaler.update(); sched.step()
    return loss.item(), bad

# --- проверка сходимости на малой подвыборке (без аугментаций): потери должны упасть заметно ниже 0,69 ---
g0 = np.random.default_rng(123)
sm = np.concatenate([g0.choice(np.where(yw == c)[0], 128, replace=False) for c in (0, 1)])
model = new_rv(0); opt, sched = make_opt(model, 8 * (len(sm) // BS)); scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
model.train(); sm_losses, tt = [], []
for ep in range(8):
    g0.shuffle(sm); ls = []
    for b in range(0, len(sm), BS):
        j = np.sort(sm[b:b + BS]); t1 = time.time()
        l, bad = step(model, opt, sched, scaler, rv_input(Xw[j]).to(DEV), torch.tensor(yw[j], device=DEV))
        if DEV == "cuda": torch.cuda.synchronize()
        tt.append(time.time() - t1); ls.append(l)
    sm_losses.append(np.mean(ls)); print(f"  проверка: эпоха {ep+1} потери {sm_losses[-1]:.3f}", flush=True)
t_step = float(np.median(tt[2:]))
del model, opt; torch.cuda.empty_cache()
assert np.isfinite(sm_losses).all() and min(sm_losses) < 0.55, f"нет сходимости: {sm_losses}"
est_run = (EPOCHS * (len(yw) / BS) * t_step * 1.1 + EPOCHS * 60) / 60   # мин (+10 % на аугментации, ~1 мин на выбор эпохи)
print(f"шаг {t_step:.2f} с | оценка одного запуска {est_run:.1f} мин")

In [ ]:
def train_run(seed, epochs=EPOCHS):
    g = np.random.default_rng(seed)
    model = new_rv(seed)
    steps_ep = int(np.ceil(len(yw) / BS))
    opt, sched = make_opt(model, epochs * steps_ep)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    best_auc, best_state, best_ep, hist, n_bad = -1, None, 0, [], 0
    t0 = time.time()
    for ep in range(epochs):
        model.train(); losses = []
        idx = g.choice(len(yw), size=len(yw), replace=True, p=w_cls)
        for b in range(0, len(idx), BS):
            j = idx[b:b + BS]
            l, bad = step(model, opt, sched, scaler, augment(rv_input(Xw[j]).to(DEV), g), torch.tensor(yw[j], device=DEV))
            n_bad += bad; losses.append(l)
        auc = roc_auc_score(y_val, predict(model, val_W))
        hist.append((round(float(np.nanmean(losses)), 3), round(auc, 3)))
        print(f"  reve запуск {seed} | эпоха {ep+1}: потери {np.nanmean(losses):.3f} | отложенные NMT {auc:.3f} | шагов с NaN/inf {n_bad} | {time.time()-t0:.0f} с", flush=True)
        if auc > best_auc:
            best_auc, best_ep, best_state = auc, ep + 1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best_ep, best_auc, hist, n_bad

TIME_LIMIT_MIN = 85
RUNS, T_TRAIN, dur = [], time.time(), []
for seed in [0, 1, 2]:
    est = np.mean(dur) if dur else est_run * 60
    if (time.time() - T_TRAIN + est) / 60 > TIME_LIMIT_MIN:
        print(f"ПРОПУСК запуск {seed}: не укладывается в {TIME_LIMIT_MIN} мин"); continue
    t0 = time.time()
    model, best_ep, best_auc, hist, n_bad = train_run(seed)
    s_ev = predict(model, [NMT[n] for n in ev.recordname]); s_tr = predict(model, TR)
    dur.append(time.time() - t0)
    RUNS.append(dict(model="reve", seed=seed, best_epoch=best_ep, val=round(best_auc, 3), hist=hist, nan_steps=n_bad,
                     nmt_eval=roc_auc_score(ev.y, s_ev), turkey=roc_auc_score(y_tr, s_tr), s_ev=s_ev, s_tr=s_tr,
                     turkey_med_p0=float(np.median(s_tr[y_tr == 0])), turkey_med_p1=float(np.median(s_tr[y_tr == 1])),
                     n_unique_tr=len(np.unique(s_tr)), minutes=round(dur[-1] / 60, 1)))
    print(f"reve запуск {seed}: эпоха {best_ep} | NMT eval {RUNS[-1]['nmt_eval']:.3f} | Турция {RUNS[-1]['turkey']:.3f} | "
          f"медиана p здоровые/эпилепсия {RUNS[-1]['turkey_med_p0']:.3f}/{RUNS[-1]['turkey_med_p1']:.3f} | "
          f"разных значений {RUNS[-1]['n_unique_tr']}/{len(s_tr)} | {dur[-1]/60:.1f} мин", flush=True)
    del model; torch.cuda.empty_cache()
print(f"всё обучение: {(time.time()-T_TRAIN)/60:.1f} мин")
runs = pd.DataFrame([{k: v for k, v in r.items() if k not in ("s_ev", "s_tr")} for r in RUNS])
runs.to_csv("/kaggle/working/reve_finetune_runs.csv", index=False)
print(runs.drop(columns="hist").round(3).to_string(index=False))

## 5. Итоги

In [ ]:
y_ev = ev.y.to_numpy()
ens_ev = np.mean([rank(r["s_ev"]) for r in RUNS], 0); ens_tr = np.mean([rank(r["s_tr"]) for r in RUNS], 0)
def fmt(y, s):
    lo, hi = boot_ci(y, s); return f"{roc_auc_score(y, s):.3f} ({lo:.2f}-{hi:.2f})"
sd = lambda v: v.std(ddof=1) if len(v) > 1 else 0.0
summary = pd.DataFrame([
    {"модель": "REVE дообученная, запуски", "n": len(runs), "NMT eval": f"{runs.nmt_eval.mean():.3f} ± {sd(runs.nmt_eval):.3f}",
     "Турция": f"{runs.turkey.mean():.3f} ± {sd(runs.turkey):.3f} ({' / '.join(f'{v:.2f}' for v in runs.turkey)})"},
    {"модель": "REVE дообученная, ансамбль запусков", "n": "", "NMT eval": fmt(y_ev, ens_ev), "Турция": fmt(y_tr, ens_tr)}])
summary.to_csv("/kaggle/working/reve_finetune_summary.csv", index=False)
print(summary.to_string(index=False))
print("Связь с амплитудой записи на Турции (Спирмен):", round(spearmanr(ens_tr, TR_AMP).correlation, 2))
d1 = pd.DataFrame({"dataset": "nmt_eval", "id": ev.recordname.tolist(), "y": y_ev}); d2 = pd.DataFrame({"dataset": "turkey", "id": tr_ids, "y": y_tr})
for d, k in [(d1, "s_ev"), (d2, "s_tr")]:
    for r in RUNS: d[f"reve_seed{r['seed']}"] = r[k]
d1["reve_ens_rank"], d2["reve_ens_rank"] = ens_ev, ens_tr
pd.concat([d1, d2]).to_csv("/kaggle/working/reve_finetune_per_person.csv", index=False)
print(f"\nвсё время блокнота: {(time.time()-T_NB)/60:.1f} мин")

## 6. Вывод Kaggle (без строк о подкачке файлов и служебных предупреждений)

Тестовые наборы считались один раз. NaN/inf нет, потери падают, у групп вероятности различны (119 разных значений из 121). Проверка сходимости на 256 окнах: потери 1,67 → 0,29 за 8 эпох.

```
позиции: (19, 3)
устройство: Tesla T4
label  abnormal  normal
loc                    
eval         90      95
train       325    1907
label  abnormal  normal
loc                    
eval         90      95
train       325    1904
окон: train 17775 | eval 10406
Турция прочитано за 131 с | эпилепсия 50, здоровые 71 | окон на запись: медиана 60
эталон перекоса: СКО сигнала записи как оценка, ROC AUC = 0.764
окон для обучения: 15103 (патология 0.14) | записей на выбор эпохи: 335
  проверка: эпоха 1 потери 1.674
  проверка: эпоха 2 потери 1.049
  проверка: эпоха 3 потери 0.652
  проверка: эпоха 4 потери 0.561
  проверка: эпоха 5 потери 0.467
  проверка: эпоха 6 потери 0.341
  проверка: эпоха 7 потери 0.312
  проверка: эпоха 8 потери 0.285
шаг 0.39 с | оценка одного запуска 26.3 мин
  reve запуск 0 | эпоха 1: потери 0.525 | отложенные NMT 0.872 | шагов с NaN/inf 0 | 207 с
  reve запуск 0 | эпоха 2: потери 0.396 | отложенные NMT 0.838 | шагов с NaN/inf 0 | 413 с
  reve запуск 0 | эпоха 3: потери 0.339 | отложенные NMT 0.855 | шагов с NaN/inf 0 | 619 с
  reve запуск 0 | эпоха 4: потери 0.293 | отложенные NMT 0.846 | шагов с NaN/inf 0 | 824 с
  reve запуск 0 | эпоха 5: потери 0.264 | отложенные NMT 0.854 | шагов с NaN/inf 0 | 1029 с
  reve запуск 0 | эпоха 6: потери 0.244 | отложенные NMT 0.857 | шагов с NaN/inf 0 | 1234 с
reve запуск 0: эпоха 1 | NMT eval 0.861 | Турция 0.853 | медиана p здоровые/эпилепсия 0.118/0.339 | разных значений 119/121 | 21.7 мин
  reve запуск 1 | эпоха 1: потери 0.533 | отложенные NMT 0.886 | шагов с NaN/inf 0 | 206 с
  reve запуск 1 | эпоха 2: потери 0.396 | отложенные NMT 0.879 | шагов с NaN/inf 0 | 412 с
  reve запуск 1 | эпоха 3: потери 0.333 | отложенные NMT 0.859 | шагов с NaN/inf 0 | 618 с
  reve запуск 1 | эпоха 4: потери 0.292 | отложенные NMT 0.854 | шагов с NaN/inf 0 | 823 с
  reve запуск 1 | эпоха 5: потери 0.245 | отложенные NMT 0.857 | шагов с NaN/inf 0 | 1027 с
  reve запуск 1 | эпоха 6: потери 0.236 | отложенные NMT 0.857 | шагов с NaN/inf 0 | 1232 с
reve запуск 1: эпоха 1 | NMT eval 0.896 | Турция 0.874 | медиана p здоровые/эпилепсия 0.070/0.172 | разных значений 119/121 | 21.7 мин
  reve запуск 2 | эпоха 1: потери 0.534 | отложенные NMT 0.882 | шагов с NaN/inf 0 | 206 с
  reve запуск 2 | эпоха 2: потери 0.395 | отложенные NMT 0.875 | шагов с NaN/inf 0 | 413 с
  reve запуск 2 | эпоха 3: потери 0.333 | отложенные NMT 0.865 | шагов с NaN/inf 0 | 619 с
  reve запуск 2 | эпоха 4: потери 0.288 | отложенные NMT 0.871 | шагов с NaN/inf 0 | 824 с
  reve запуск 2 | эпоха 5: потери 0.252 | отложенные NMT 0.861 | шагов с NaN/inf 0 | 1028 с
  reve запуск 2 | эпоха 6: потери 0.239 | отложенные NMT 0.861 | шагов с NaN/inf 0 | 1233 с
reve запуск 2: эпоха 1 | NMT eval 0.846 | Турция 0.862 | медиана p здоровые/эпилепсия 0.173/0.479 | разных значений 119/121 | 21.7 мин
всё обучение: 65.1 мин
model  seed  best_epoch   val  nan_steps  nmt_eval  turkey  turkey_med_p0  turkey_med_p1  n_unique_tr  minutes
 reve     0           1 0.872          0     0.861   0.853          0.118          0.339          119     21.7
 reve     1           1 0.886          0     0.896   0.874          0.070          0.172          119     21.7
 reve     2           1 0.882          0     0.846   0.862          0.173          0.479          119     21.7
                             модель n          NMT eval                             Турция
          REVE дообученная, запуски 3     0.868 ± 0.026 0.863 ± 0.011 (0.85 / 0.87 / 0.86)
REVE дообученная, ансамбль запусков   0.875 (0.83-0.92)                  0.874 (0.80-0.93)
Связь с амплитудой записи на Турции (Спирмен): 0.31
всё время блокнота: 71.4 мин
```


## 7. Ансамбли с CBraMod и DeepEpilepsy (по людям, `results/reve_ensembles_per_person.csv`)

Люди сопоставлены по идентификатору записи; в NMT eval у блокнота CBraMod/DeepEpilepsy 184 записи, у REVE 185 — общие 184 (+121 в Турции), ранги пересчитаны на общем наборе, поэтому у REVE 0,879 вместо 0,875.

| Модель / ансамбль (средние ранги) | NMT eval (95% ДИ) | Турция (95% ДИ) |
|---|---|---|
| **REVE дообученная (3 запуска)** | 0,879 (0,83–0,92) | **0,874 (0,80–0,93)** |
| CBraMod | 0,845 (0,79–0,90) | 0,820 (0,74–0,88) |
| DeepEpilepsy | 0,900 (0,85–0,94) | 0,682 (0,59–0,77) |
| REVE + CBraMod | 0,875 (0,82–0,92) | 0,872 (0,80–0,93) |
| REVE + DeepEpilepsy | **0,905 (0,86–0,94)** | 0,833 (0,75–0,90) |
| CBraMod + DeepEpilepsy | 0,893 (0,85–0,93) | 0,794 (0,71–0,87) |
| Все три | 0,896 (0,85–0,93) | 0,856 (0,78–0,92) |

Запуски REVE: NMT eval 0,86 / 0,90 / 0,85 (0,868 ± 0,026), Турция 0,85 / 0,87 / 0,86 (0,863 ± 0,011).

Парный бутстреп разности ROC AUC (95% ДИ):
- REVE − CBraMod: NMT eval +0,034 (−0,002…+0,071); **Турция +0,055 (−0,008…+0,119)** — ДИ включает 0.
- REVE − DeepEpilepsy: NMT eval −0,021 (−0,061…+0,017); Турция +0,193 (+0,090…+0,291).
- REVE + CBraMod − CBraMod: NMT eval +0,030 (+0,010…+0,052); Турция +0,052 (+0,017…+0,091).
- REVE + CBraMod − REVE: NMT eval −0,004, Турция −0,002 — ансамбль ничего не добавляет к REVE.
- Три модели − REVE + CBraMod: NMT eval +0,021 (+0,005…+0,038); Турция −0,016 (−0,053…+0,019).

Согласие оценок (Спирмен), Турция: REVE–CBraMod 0,72, REVE–DeepEpilepsy 0,39, CBraMod–DeepEpilepsy 0,40.

## 8. Выводы

1. **REVE после дообучения — лучшая одиночная модель на Турции**: 0,87 (0,80–0,93) против 0,82 у CBraMod и 0,68 у DeepEpilepsy, запуски 0,85–0,87 (разброс меньше, чем у CBraMod). Преимущество над CBraMod **не доказано**: разность +0,055, ДИ −0,008…+0,119. Над DeepEpilepsy — значимо. На NMT eval все три близки (0,85–0,90).
2. **Ансамбль с CBraMod REVE не улучшает** (0,872 против 0,874) — их оценки согласуются сильно (0,72). Полезнее пара REVE + DeepEpilepsy на NMT eval (0,905), но на Турции она хуже одиночной REVE (0,83); все три — 0,86 на Турции и 0,90 на NMT eval, то есть ансамбль скорее «страховка», чем выигрыш.
3. **Лучшая эпоха во всех трёх запусках — первая**: после неё отложенный AUC падает (0,87–0,89 → 0,84–0,86), хотя потери на обучении идут вниз (0,53 → 0,24) — модель быстро запоминает обучающие записи. Значит, нужно меньше шагов (1–2 эпохи) и/или меньшая скорость обучения основы, замороженные первые слои, больше аугментаций. Это следующий шаг, в этом блокноте не перезапускалось. Отложенные 15 % выбирали эпоху, так что цифры по NMT eval и Турции честные, но запуски на самом деле почти «одноэпохные».
4. **Связь с амплитудой записи на Турции выше, чем у CBraMod**: Спирмен +0,31 против +0,12 (одна амплитуда даёт 0,76). Часть 0,87 может быть от косвенных признаков, различающихся между группами (амплитуда, возраст, аппарат); нужна проверка, например поканальная нормировка или контроль по амплитуде.
5. **Выбор окна**: 10 с вместо 30 с (при 30 с одно окно — 19·33 ≈ 630 токенов, оценка > 1 ч на запуск), до 8 окон на запись NMT train (15 103 окна, как у образца по числу окон, но каждое втрое короче: ≈80 с на запись вместо ≈240 с). Для NMT eval и Турции — до 60 окон 10 с на запись, равномерно (10 406 окон NMT eval), медиана вероятности. Это отличается от CBraMod/DeepEpilepsy (30 с), так что сравнение не абсолютно одинаковое по условиям; шаг обучения 0,39 с, запуск 21,7 мин.
6. **Важная оговорка про данные**: REVE предобучена на 92 наборах, в том числе на TUH/TUEG (так же, как CBraMod и LaBraM). Для проверки на TUEP нужно исключить пересечение пациентов; на NMT и Турции пересечения предобучения неизвестны — полностью исключить утечку признаков нельзя.
7. Прочие оговорки: 3 запуска; Турция — 121 человек, ДИ широкие; NMT («патология») и Турция («эпилепсия против здоровых») — разные задачи; фильтр 0,3–75 Гц + режекция 50 Гц как у CBraMod.

**Затраты:** 71 мин блокнот (обучение 65 мин) ≈ 1,2 ч GPU T4 (квота Kaggle: 14,2 → 15,4 ч из 30; ограничение 20 ч соблюдено).

Файлы: `results/reve_finetune_{runs,summary,per_person}.csv`, `results/reve_ensembles_{summary,per_person}.csv`.
